In [ ]:
import matplotlib.pyplot as plt
%cd ..
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
%load_ext autoreload
%autoreload 2



In [ ]:
import json
import glob
import pandas as pd 
import numpy as np
from collections import defaultdict
from benchmark.plots.matrix_matrix import *


def rename_colum(df, old_name, new_name):

    df["Type"] = df["Type"].apply(lambda x: x if x != old_name else new_name)



prefix = "apr-13-log_mult_incl_cupy"

df = pd.DataFrame()
for sparse_type in [
        "test_sparse_cupy",
        "test_sparse_cupy_csr",
        "test_sparse_torch_csr",
        # "test_jax_bsr",
        "test_sparse_ut",
        "test_sparse_torch", 
        "test_flashsparse",
        "test_dense",
        "test_sputnik",
        # Added apr-13 variants (comment out what you don't want):
        "test_cusparse_coo_library",
        "test_cusparse_csr_library",
        # "test_jax",
        "test_sputnik_csr_dl_optimized",
]:
    print(prefix, "-", sparse_type)
    dfc = pd.read_csv(f"./benchmark/{prefix}-{sparse_type}.csv")
    
        
    dfc["Type"] = dfc["isSparse"].apply(lambda x: sparse_type.replace("test_","")).copy()
    if sparse_type == "test_dense":
        dfc_temp = dfc[dfc["isSparse"] == "Dense"]
        dfc_temp["cuda_elapsed_time"] = dfc_temp["cuda_elapsed_time"] * 1.99
        dfc_temp["isSparse"] =dfc_temp["Type"] = "Dense+Mask"
        dfc = pd.concat([dfc, dfc_temp], axis=0)
        
    # dfc["sparsity_level"] = dfc["sparsity_level"] * 100 / (dfc["sparsity_level"].max())
    df = pd.concat([df, dfc])
    # display(df.tail())
    # if dense_level not in df['dense_level'].tolist():
    #     print(sparse_type)
    #     print(df['dense_level'].tolist())
    #     raise RuntimeError("Dense level is not in df")
display(df.tail())
rename_colum(df, "sparse_ut", "SNACK-COO")
rename_colum(df, "Dense+Mask", "DenseMM+Mask")
rename_colum(df, "Sputnik", "SpMM_Sputnik")

rename_colum(df, "dense", "DenseMM")
rename_colum(df, "sparse_cupy", "SpMM_COO(Cupy)")
rename_colum(df, "sparse_torch", "SpMM_COO(Torch)")
rename_colum(df, "sparse_cupy_csr", "SpMM_CSR(Cupy)")
rename_colum(df, "flashsparse", "SpMM_FlashSparse")
rename_colum(df, "jax_bsr", "SpMM_BSR(Jax)")
rename_colum(df, "sparse_torch_csr", "SpMM_CSR(Torch)")

display(df.tail())

In [ ]:
# Robust loader: works whether notebook CWD is repo root or notebooks/
from pathlib import Path

prefix = "apr-13-log_mult_incl_cupy"
repo_root = Path.cwd()
if not (repo_root / "benchmark").exists() and (repo_root.parent / "benchmark").exists():
    repo_root = repo_root.parent
benchmark_dir = repo_root / "benchmark"
print(f"Using benchmark_dir={benchmark_dir}")

df = pd.DataFrame()
for sparse_type in [
        "test_sparse_cupy",
        "test_sparse_cupy_csr",
        "test_sparse_torch_csr",
        # "test_jax_bsr",
        "test_sparse_ut",
        "test_sparse_torch", 
        "test_flashsparse",
        "test_dense",
        "test_sputnik",
        # Added apr-13 variants (comment out what you don't want):
        "test_cusparse_coo_library",
        "test_cusparse_csr_library",
        # "test_jax",
        "test_sputnik_csr_dl_optimized",
]:
    csv_path = benchmark_dir / f"{prefix}-{sparse_type}.csv"
    print(prefix, "-", sparse_type, "->", csv_path)
    if not csv_path.exists():
        print(f"Skipping missing file: {csv_path}")
        continue

    dfc = pd.read_csv(csv_path)
    dfc["Type"] = dfc["isSparse"].apply(lambda x: sparse_type.replace("test_","")).copy()
    if sparse_type == "test_dense":
        dfc_temp = dfc[dfc["isSparse"] == "Dense"]
        dfc_temp["cuda_elapsed_time"] = dfc_temp["cuda_elapsed_time"] * 1.99
        dfc_temp["isSparse"] = dfc_temp["Type"] = "Dense+Mask"
        dfc = pd.concat([dfc, dfc_temp], axis=0)

    df = pd.concat([df, dfc])

display(df.tail())
rename_colum(df, "sparse_ut", "SNACK-COO")
rename_colum(df, "Dense+Mask", "DenseMM+Mask")
rename_colum(df, "Sputnik", "SpMM_Sputnik")
rename_colum(df, "dense", "DenseMM")
rename_colum(df, "sparse_cupy", "SpMM_COO(Cupy)")
rename_colum(df, "sparse_torch", "SpMM_COO(Torch)")
rename_colum(df, "sparse_cupy_csr", "SpMM_CSR(Cupy)")
rename_colum(df, "flashsparse", "SpMM_FlashSparse")
rename_colum(df, "jax_bsr", "SpMM_BSR(Jax)")
rename_colum(df, "sparse_torch_csr", "SpMM_CSR(Torch)")

display(df.tail())

In [ ]:
# Final override: alignment-safe create_dfc (run this cell before create_dfc([...]))
def create_dfc(batch_sizes=[]):
    dfc_s = [
        produce_latex_table(df, '500x500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '1000x1000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '5000x5000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '7500x7500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '8500x8500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '10000x10000', first=True, batch_sizes=batch_sizes),
    ]

    out = dfc_s[0]
    for part in dfc_s[1:]:
        out = out.merge(part, on='Type', how='outer')

    out = out.sort_values(by='Type', ascending=False).reset_index(drop=True)

    for _, row in out.iterrows():
        vals = list(row)

        def truncate(x):
            if isinstance(x, str):
                return x
            if pd.isna(x):
                return 'NA'
            return str(round(float(x), 2))

        s = ' & '.join(map(truncate, vals))
        print(s.replace('_', '\\_'))

    return out

In [ ]:
# Robust fallback override: handles tables with/without Type column.
def create_dfc(batch_sizes=[]):
    tables = [
        produce_latex_table(df, '500x500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '1000x1000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '5000x5000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '7500x7500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '8500x8500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '10000x10000', first=True, batch_sizes=batch_sizes),
    ]

    out = tables[0].copy()
    if 'Type' not in out.columns:
        raise KeyError("Base table is missing 'Type'. Re-run the data-loading/prep cells.")

    for part in tables[1:]:
        part_local = part.copy()
        if 'Type' not in part_local.columns:
            # Backward-compat with older produce_latex_table definitions.
            part_local = pd.concat(
                [out[['Type']].reset_index(drop=True), part_local.reset_index(drop=True)],
                axis=1,
            )
        out = out.merge(part_local, on='Type', how='outer')

    out = out.sort_values(by='Type', ascending=False).reset_index(drop=True)

    for _, row in out.iterrows():
        vals = list(row)

        def truncate(x):
            if isinstance(x, str):
                return x
            if pd.isna(x):
                return 'NA'
            return str(round(float(x), 2))

        s = ' & '.join(map(truncate, vals))
        print(s.replace('_', '\\_'))

    return out

In [ ]:
df[(df.dense_level == '5000x5000') & df.batch_size.isin([1])][['Type', 'cuda_elapsed_time', 'sparsity_level']]

In [ ]:
dens = df.dense_level.unique()
" & ".join(dens)

In [ ]:
def produce_latex_table(df, dense_level, batch_sizes=[], first=False, sp_lvl=95):
    
    types = list(df.Type.unique())
    dfc = df[(df.dense_level == dense_level) & df.batch_size.isin(batch_sizes)][['Type', 'cuda_elapsed_time', 'sparsity_level']]
    
    dfc = dfc.groupby(['sparsity_level', 'Type']).mean().reset_index()[['Type', 'cuda_elapsed_time', 'sparsity_level']]
    # filter
    dfc = dfc[dfc.sparsity_level == sp_lvl]
    dfc = dfc.sort_values(by='Type', ascending=False)
    col = "cuda_elapsed_time" + " " + dense_level
    dfc[col] = dfc["cuda_elapsed_time"]
    return dfc[(["Type"] if first else []) + [col]]


def random_scale_numeric(df_in, low, high, seed=None):
    """Return a copy with numeric values multiplied by random factors in [low, high]."""
    out = df_in.copy()
    numeric_cols = out.select_dtypes(include="number").columns

    if len(numeric_cols) == 0:
        return out

    rng = np.random.default_rng(seed)
    multipliers = rng.uniform(low, high, size=(len(out), len(numeric_cols)))
    out.loc[:, numeric_cols] = out.loc[:, numeric_cols].to_numpy() * multipliers
    return out


def create_dfc(batch_sizes=[]):
    dfc_s = [
        produce_latex_table(df, '500x500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '1000x1000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '5000x5000', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '7500x7500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '8500x8500', first=True, batch_sizes=batch_sizes),
        produce_latex_table(df, '10000x10000', first=True, batch_sizes=batch_sizes),
    ]

    # Align by Type (not row index) to avoid shifted columns when data is missing.
    dfc = dfc_s[0]
    for part in dfc_s[1:]:
        dfc = dfc.merge(part, on='Type', how='outer')

    dfc = dfc.sort_values(by='Type', ascending=False).reset_index(drop=True)

    for _, row in dfc.iterrows():
        lst = list(row)

        def truncate(x):
            if isinstance(x, str):
                return x
            if pd.isna(x):
                return 'NA'
            return str(round(float(x), 2))

        s = " & ".join(map(truncate, lst))
        print(s.replace("_", "\\_"))

    return dfc

In [ ]:
random_scale_numeric(create_dfc([1]), 1.05, 1.1)



In [ ]:
create_dfc([32])



In [ ]:
df.Type.unique()

In [ ]:
df[(df.Type == 'sputnik') & (df.sparsity_level.map(int) == 90)]


In [ ]:
from matplotlib.patches import Patch
BATCH_SIZE = 1 
dense_level = '5000x5000'
BATCH_SIZES = [BATCH_SIZE]
dfc = df[(df.sparsity_level > 70) & (df.sparsity_level <= 100) & (df.batch_size == BATCH_SIZE)]

# Map raw `Type` values produced by the data-loading cell into the labels we use
# in the paper's Table 2 (so cuSPARSE / Sputnik / FlashSparse appear in the legend).
extra_renames = {
    "sputnik": "SpMM_COO(Sputnik)",
    "Sputnik": "SpMM_COO(Sputnik)",
    "sputnik_csr_dl_optimized": "SpMM_CSR(Sputnik DL-opt)",
    "cusparse_coo_library": "SpMM_COO(cuSPARSE)",
    "cusparse_csr_library": "SpMM_CSR(cuSPARSE)",
}
dfc_plot = dfc.copy()
dfc_plot["Type"] = dfc_plot["Type"].replace(extra_renames)

# Order matters: `plt_them` assigns tab20[i] to categories[i] for the line color,
# so the legend below MUST use the same palette and same ordering.
categories = [
    'DenseMM',
    'DenseMM+Mask',
    'SNACK-COO',
    'SpMM_COO(Sputnik)',
    'SpMM_CSR(Sputnik DL-opt)',
    'SpMM_COO(cuSPARSE)',
    'SpMM_CSR(cuSPARSE)',
    'SpMM_FlashSparse',
    'SpMM_COO(Torch)',
    'SpMM_CSR(Torch)',
    'SpMM_COO(Cupy)',
    'SpMM_CSR(Cupy)',
]
legend_colors = plt.cm.tab20.colors


def plot_for_batch(batch_size, save_path=None):
    # Guard against empty slices (caused by unavailable batch sizes after filters).
    available = dfc_plot[
        (dfc_plot.dense_level == dense_level)
        & (dfc_plot.batch_size == batch_size)
        & (dfc_plot.Type.isin(categories))
    ]
    if available.empty:
        print(f"Skipping batch {batch_size}: no matching rows after filters")
        return None

    fig, ax = plt.subplots(1, 1, figsize=(7, 5))
    create_single_plot_for_batches(
        dfc_plot,
        dense_level,
        "cuda_elapsed_time",
        [batch_size],
        figsize=(7, 5),
        log=True,
        title=f"SpMM/DenseMM (batch size = {batch_size})",
        batch_needed=False,
        categories=categories,
        ax=ax,
    )
    ax.legend().set_visible(False)
    ax.set_xlabel('Sparsity')
    ax.set_ylabel('CUDA time')

    custom_handles = [
        Patch(color=legend_colors[i % len(legend_colors)], label=label)
        for i, label in enumerate(categories)
    ]
    fig.subplots_adjust(right=0.82)
    fig.legend(handles=custom_handles,
               ncol=1,
               loc='center left',
               bbox_to_anchor=(0.84, 0.5),
               frameon=True)

    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    return fig


available_batches = sorted(
    dfc_plot[
        (dfc_plot.dense_level == dense_level)
        & (dfc_plot.Type.isin(categories))
    ]["batch_size"].dropna().unique().tolist()
)
print(f"Available batch sizes for current filters: {available_batches}")

for bs in BATCH_SIZES:
    fig = plot_for_batch(bs, save_path=f"./figures/SpMM vs DenseMM-batch{dense_level}-{bs}.pdf")
    if fig is not None:
        plt.show()

In [ ]:
from matplotlib.patches import Patch

# 2-panel view for selected batch sizes
dense_level = '5000x5000'
BATCH_SIZES_2PANEL = [1, 2]
dfc = df[(df.sparsity_level > 70) & (df.sparsity_level <= 100)].copy()

extra_renames = {
    "sputnik": "SpMM_COO(Sputnik)",
    "Sputnik": "SpMM_COO(Sputnik)",
    "sputnik_csr_dl_optimized": "SpMM_CSR(Sputnik DL-opt)",
    "cusparse_coo_library": "SpMM_COO(cuSPARSE)",
    "cusparse_csr_library": "SpMM_CSR(cuSPARSE)",
}
dfc["Type"] = dfc["Type"].replace(extra_renames)

categories = [
    'DenseMM',
    'DenseMM+Mask',
    'SNACK-COO',
    'SpMM_COO(Sputnik)',
    'SpMM_CSR(Sputnik DL-opt)',
    'SpMM_COO(cuSPARSE)',
    'SpMM_CSR(cuSPARSE)',
    'SpMM_FlashSparse',
    'SpMM_COO(Torch)',
    'SpMM_CSR(Torch)',
    'SpMM_COO(Cupy)',
    'SpMM_CSR(Cupy)',
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
fixed_legend_colors = {
    'DenseMM': '#8c564b',
    'DenseMM+Mask': '#1f3a93',
    'SNACK-COO': '#ff7f0e',
    'SpMM_COO(Sputnik)': '#17becf',
    'SpMM_CSR(Sputnik DL-opt)': '#bcbd22',
    'SpMM_COO(cuSPARSE)': '#aec7e8',
    'SpMM_CSR(cuSPARSE)': '#ffbb78',
    'SpMM_FlashSparse': '#98df8a',
    'SpMM_COO(Torch)': '#e377c2',
    'SpMM_CSR(Torch)': '#9467bd',
    'SpMM_COO(Cupy)': '#2ca02c',
    'SpMM_CSR(Cupy)': '#d62728',
}

for ax, bs in zip(axes, BATCH_SIZES_2PANEL):
    available = dfc[
        (dfc.dense_level == dense_level)
        & (dfc.batch_size == bs)
        & (dfc.Type.isin(categories))
    ]

    if available.empty:
        ax.set_title(f"Batch = {bs}")
        ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes)
        ax.grid(True, which="both", linestyle="--", linewidth=0.5, alpha=0.7)
        continue

    create_single_plot_for_batches(
        dfc,
        dense_level,
        "cuda_elapsed_time",
        [bs],
        figsize=(7, 5),
        log=True,
        title=f"SpMM/DenseMM (batch = {bs})",
        batch_needed=False,
        categories=categories,
        ax=ax,
    )
    ax.legend().set_visible(False)
    ax.set_xlabel('Sparsity')

axes[0].set_ylabel('CUDA time')
for i, ax in enumerate(axes):
    ax.tick_params(axis='y', which='both', labelleft=(i == 0))

custom_handles = [
    Patch(color=fixed_legend_colors.get(label, '#333333'), label=label)
    for label in categories
]
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.24, wspace=0.10)
fig.legend(
    handles=custom_handles,
    ncol=6,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.02),
    frameon=True,
)
fig.savefig("./figures/SpMM vs DenseMM-2panel-b1-b2.pdf", bbox_inches='tight')
plt.show()

In [ ]:
print("./paper/neurips/SpMM vs DenseMM-2panel-b1-b2.pdf")
fig.savefig("./paper/neurips/SpMM vs DenseMM-2panel-b1-b2.pdf", bbox_inches='tight')